# Kayak Project — ETL (Data Lake → Data Warehouse)

This notebook stores the clean files in a Data Lake, then loads them into a SQL Data Warehouse, as the brief asks.

The Data Lake is a Supabase Storage bucket with an S3-compatible API. The Data Warehouse is a PostgreSQL database on Supabase. The AWS account used during the course is no longer accessible, so the same architecture runs on Supabase.

Steps:
- Connect to the database
- Add the `city_id` key to the clean files
- Build the enriched file: each hotel with the weather of its city
- Upload the three files to the Data Lake
- Replace the tables
- Read the files back from the Data Lake
- Transform: remove the parameters from the Booking addresses
- Load the tables with `COPY`
- Check the result

The next cell imports the libraries.

In [1]:
from botocore.config import Config
from dotenv import load_dotenv
from io import BytesIO, StringIO
import os
import boto3
import pandas as pd
import psycopg2

## Database Connection

Connects to PostgreSQL through the Supabase session pooler, which accepts IPv4 (the direct connection uses IPv6 by default). The credentials come from the `.env` file and stay out of the code. The output confirms the connection.

In [2]:
load_dotenv()

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    sslmode="require"
)

print("Connected ✅")

Connected ✅


## Add the City Key and Build the Enriched File

Adds `city_id`, read from the raw geocoded file, to both clean files. It becomes the primary key of `cities_weather` and a foreign key in `hotels`. Then it joins each hotel to the weather of its city: this is the enriched file the brief asks for. Nothing is collected again, and the cell can run several times. The output shows 34 cities and 511 hotels, with 9 columns each.

In [3]:
city_ids = (
    pd.read_csv("../data/raw/cities_geocoded.csv")[["city_id", "city_name"]]
    .rename(columns={"city_name": "city"})
)

cities_df = (
    pd.read_csv("../data/clean/cities_weather.csv")
    .drop(columns="city_id", errors="ignore")
    .merge(city_ids, on="city", how="left", validate="one_to_one")
)
hotels_df = (
    pd.read_csv("../data/clean/hotels.csv")
    .drop(columns="city_id", errors="ignore")
    .merge(city_ids, on="city", how="left", validate="many_to_one")
)
assert cities_df["city_id"].notna().all() and hotels_df["city_id"].notna().all()

cities_df = cities_df[["city_id"] + [c for c in cities_df.columns if c != "city_id"]]
hotels_df = hotels_df[["city_id"] + [c for c in hotels_df.columns if c != "city_id"]]

cities_df.to_csv("../data/clean/cities_weather.csv", index=False)
hotels_df.to_csv("../data/clean/hotels.csv", index=False)

hotel_columns = {"latitude": "hotel_latitude", "longitude": "hotel_longitude"}
enriched_df = cities_df.merge(hotels_df.drop(columns="city").rename(columns=hotel_columns), on="city_id", how="left", validate="one_to_many")
enriched_df.to_csv("../data/clean/cities_hotels_enriched.csv", index=False)

print("Cities:", cities_df.shape, "| Hotels:", hotels_df.shape)

Cities: (34, 9) | Hotels: (511, 9)


## Upload Data to the Data Lake

Uploads the three files to the bucket: weather, hotels and the enriched file. The bucket speaks the S3 protocol, so we use boto3, Amazon's library, with the address of the Supabase server. All the settings come from the `.env` file. The output confirms the upload.

In [4]:
s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("S3_ENDPOINT_URL"),
    region_name=os.getenv("S3_REGION"),
    aws_access_key_id=os.getenv("S3_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("S3_SECRET_ACCESS_KEY"),
    config=Config(
        s3={"addressing_style": "path"},
        request_checksum_calculation="when_required",
        response_checksum_validation="when_required",
    ),
)
bucket_name = os.getenv("S3_BUCKET")

s3.upload_file("../data/clean/cities_weather.csv", bucket_name, "cities_weather.csv")
s3.upload_file("../data/clean/hotels.csv", bucket_name, "hotels.csv")
s3.upload_file("../data/clean/cities_hotels_enriched.csv", bucket_name, "cities_hotels_enriched.csv")

print("Upload to the data lake successful ✅")

Upload to the data lake successful ✅


## Create Tables

Defines the Data Warehouse:

- `cities_weather`: one row per city, with its average weather
- `hotels`: one row per hotel, from Booking
- `cities_hotels_enriched`: a view that rebuilds the enriched file of the Data Lake

The old tables are dropped first, so that running the notebook again gives the same result, with no duplicate rows. The output confirms that the tables are recreated, but not committed yet.

In [5]:
cursor = conn.cursor()

# the view first: it depends on both tables
cursor.execute("DROP VIEW IF EXISTS cities_hotels_enriched;")
# hotels next: it references cities_weather
cursor.execute("DROP TABLE IF EXISTS hotels;")
cursor.execute("DROP TABLE IF EXISTS cities_weather;")

cursor.execute("""
CREATE TABLE cities_weather (
    city_id INTEGER PRIMARY KEY,
    city TEXT,
    latitude FLOAT,
    longitude FLOAT,
    temperature FLOAT,
    humidity FLOAT,
    rain_prob FLOAT,
    wind_speed FLOAT,
    weather_score FLOAT
);
""")

cursor.execute("""
CREATE TABLE hotels (
    city_id INTEGER REFERENCES cities_weather (city_id),
    city TEXT,
    hotel_name TEXT,
    score TEXT,
    url TEXT,
    description TEXT,
    score_clean FLOAT,
    latitude FLOAT,
    longitude FLOAT
);
""")

# Extra layer of protection, in addition to the disabled Data API
cursor.execute("ALTER TABLE cities_weather ENABLE ROW LEVEL SECURITY;")
cursor.execute("ALTER TABLE hotels ENABLE ROW LEVEL SECURITY;")

# security_invoker: the view applies the row level security of the tables
cursor.execute("""
CREATE VIEW cities_hotels_enriched WITH (security_invoker = true) AS
SELECT
    c.city_id, c.city, c.latitude, c.longitude, c.temperature, c.humidity, c.rain_prob, c.wind_speed, c.weather_score,
    h.hotel_name, h.score, h.url, h.description, h.score_clean,
    h.latitude AS hotel_latitude, h.longitude AS hotel_longitude
FROM cities_weather c
LEFT JOIN hotels h ON h.city_id = c.city_id;
""")

print("Tables recreated, not committed yet")

Tables recreated, not committed yet


## Read the Data Lake

Reads the two source files back from the Data Lake, because the Data Warehouse must be loaded from the lake, not from the local files. No output.

In [6]:
def read_csv_from_s3(key):
    response = s3.get_object(Bucket=bucket_name, Key=key)
    return pd.read_csv(BytesIO(response["Body"].read()))


cities_s3 = read_csv_from_s3("cities_weather.csv")
hotels_s3 = read_csv_from_s3("hotels.csv")

## Transform: Remove the URL Parameters

Keeps only the address of each hotel page, by cutting each Booking address at the "?". What comes after describes the browsing session, not the hotel. The output counts the addresses with parameters, before and after this step: 0 and 0, because the clean files already have none. The step protects the Data Warehouse if a new collection brings them back.

In [7]:
def strip_url_params(urls):
    return urls.str.split("?").str[0]


lake_with_params = hotels_s3["url"].str.contains("?", regex=False).sum()
hotels_s3["url"] = strip_url_params(hotels_s3["url"])
print("Addresses with parameters in the data lake:", lake_with_params, "| after the transformation:", hotels_s3["url"].str.contains("?", regex=False).sum())

Addresses with parameters in the data lake: 0 | after the transformation: 0


## Load the Database

Loads the two tables with the `COPY` command, faster than inserting row by row. Dropping the tables, creating them and loading the data form one transaction: if a step fails before the commit, the previous tables stay in place. The output confirms the commit.

In [8]:
def copy_to_table(frame, table):
    buffer = StringIO()
    frame.to_csv(buffer, index=False, header=False)
    buffer.seek(0)
    columns = ", ".join(frame.columns)
    cursor.copy_expert(f"COPY {table} ({columns}) FROM STDIN WITH CSV", buffer)


copy_to_table(cities_s3, "cities_weather")
copy_to_table(hotels_s3, "hotels")

conn.commit()
print("Loaded from the data lake and committed ✅")

Loaded from the data lake and committed ✅


## Validation

Checks that the Data Warehouse holds the same data as the Data Lake. It compares the row counts, checks that every hotel is linked to its city and that no address keeps parameters, and compares the view with the enriched file, value by value. The text export of this connection rounds floats to 15 digits, so `extra_float_digits` is set to get the full precision. The output shows 34 cities, 511 hotels, 511 hotels matched to their city, and a view identical to the file.

In [9]:
cursor.execute("SELECT COUNT(*) FROM cities_weather;")
n_cities = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM hotels;")
n_hotels = cursor.fetchone()[0]
cursor.execute("""
SELECT COUNT(*)
FROM hotels h
JOIN cities_weather c ON h.city_id = c.city_id AND h.city = c.city;
""")
n_joined = cursor.fetchone()[0]

print("Cities in the database:", n_cities, "| in the data lake:", len(cities_s3))
print("Hotels in the database:", n_hotels, "| in the data lake:", len(hotels_s3))
print("Hotels matched to their city:", n_joined)

cursor.execute("SELECT COUNT(*) FROM hotels WHERE url LIKE '%?%';")
n_with_params = cursor.fetchone()[0]
print("Addresses with parameters in the database:", n_with_params)

assert n_cities == len(cities_s3) and n_hotels == len(hotels_s3) and n_joined == n_hotels and n_with_params == 0

# The view must give back the enriched file of the data lake
enriched_s3 = read_csv_from_s3("cities_hotels_enriched.csv")
print("Addresses with parameters in the enriched file of the data lake:", enriched_s3["url"].str.contains("?", regex=False).sum())
enriched_s3["url"] = strip_url_params(enriched_s3["url"])
buffer = StringIO()
# Full precision in the text export: this connection rounds floats to 15 digits otherwise
cursor.execute("SET extra_float_digits = 3;")
cursor.copy_expert(f"COPY (SELECT {', '.join(enriched_s3.columns)} FROM cities_hotels_enriched) TO STDOUT WITH CSV HEADER", buffer)
buffer.seek(0)
enriched_db = pd.read_csv(buffer)

sort_key = ["city_id", "url"]
same_data = (
    enriched_db.sort_values(sort_key).reset_index(drop=True)
    .equals(enriched_s3.sort_values(sort_key).reset_index(drop=True))
)
print("Enriched rows in the view:", len(enriched_db), "| in the data lake:", len(enriched_s3), "| identical:", same_data)
assert same_data

Cities in the database: 34 | in the data lake: 34
Hotels in the database: 511 | in the data lake: 511
Hotels matched to their city: 511
Addresses with parameters in the database: 0


Addresses with parameters in the enriched file of the data lake: 0
Enriched rows in the view: 511 | in the data lake: 511 | identical: True


Lists the objects of the database. The output shows the two tables, `cities_weather` and `hotels`, and the view `cities_hotels_enriched`.

In [10]:
cursor.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public';
""")

print(cursor.fetchall())

[('cities_weather',), ('cities_hotels_enriched',), ('hotels',)]


Runs a simple SQL query on `hotels` and shows 5 rows, to see the data as the Data Warehouse returns it. The output shows the `city_id` key next to each hotel, here 19 for Bormes les Mimosas.

In [11]:
query = "SELECT * FROM hotels LIMIT 5;"
cursor.execute(query)
columns = [column[0] for column in cursor.description]
df = pd.DataFrame(cursor.fetchall(), columns=columns)

df.head()

,city_id,city,hotel_name,score,url,description,score_clean,latitude,longitude
0,19,Bormes les Mimosas,"Au Levant, Maison de village de charme, vue mer",None,https://www.booking.com/hotel/fr/au-levant-mai...,"L’hébergement Au Levant, Maison de village de ...",NaN,43.150661,6.341298
1,19,Bormes les Mimosas,Linaya Hôtel & Restaurant,"Avec une note de 7,47,4Bien 60 expériences vé...",https://www.booking.com/hotel/fr/linaya-amp-re...,L’établissement Linaya Hôtel & Restaurant se s...,7.4,43.121742,6.356741
2,19,Bormes les Mimosas,Hostellerie du Cigalou,"Avec une note de 8,88,8Superbe 461 expériences...",https://www.booking.com/hotel/fr/hostellerie-d...,Cet hôtel 3 étoiles est situé dans le village ...,8.8,43.151958,6.343283
3,19,Bormes les Mimosas,Hôtel de la Plage - HDLP,"Avec une note de 9,19,1Fabuleux 281 expérienc...",https://www.booking.com/hotel/fr/de-la-plage-h...,L’établissement Hôtel de la Plage - HDLP se si...,9.1,43.124617,6.357572
4,19,Bormes les Mimosas,Studio climatisé avec une place de parking et ...,"Avec une note de 8,98,9Superbe 38 expériences ...",https://www.booking.com/hotel/fr/studio-cosy-c...,L’hébergement Studio climatisé avec une place ...,8.9,43.122825,6.356860


Supabase Storage is the Data Lake: it keeps the clean files. Supabase PostgreSQL is the Data Warehouse: its tables can be queried with SQL.